<a href="https://colab.research.google.com/github/irene-ch-yeh/colab/blob/main/ntuhw2_wistron_irene_Pb1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install dwave-neal dwave-ocean-sdk dimod

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import time
from itertools import combinations

seed = 11412044
np.random.seed(seed)

weights = [23, 31, 29, 44, 53, 38, 63, 85, 89, 82]
values  = [92, 57, 49, 68, 60, 43, 67, 84, 87, 72]
W = 165
n = 10

best_value = 0
best_combo = []
best_weight = 0

start_classical = time.time()
for r in range(n + 1):
    for combo in combinations(range(n), r):
        tw = sum(weights[i] for i in combo)
        tv = sum(values[i] for i in combo)
        if tw <= W and tv > best_value:
            best_value = tv
            best_combo = list(combo)
            best_weight = tw
time_classical = time.time() - start_classical

print("=== Classical Brute Force ===")
print(f"Optimal items (1-indexed): {[i+1 for i in best_combo]}")
print(f"Total weight: {best_weight}")
print(f"Total value:  {best_value}")
print(f"Time: {time_classical:.6f} s")

=== Classical Brute Force ===
Optimal items (1-indexed): [1, 2, 3, 4, 6]
Total weight: 165
Total value:  309
Time: 0.001724 s


In [ ]:
# ============================================================
#  QUBO 推導
# ============================================================
import dimod
import neal

def build_knapsack_qubo(weights, values, W, lam):
    """Build QUBO dict for 0-1 knapsack with slack variables."""
    n = len(weights)
    M = int(np.ceil(np.log2(W)))  # 8 slack bits
    Q = {}

    # --- 物品對角線 ---
    for i in range(n):
        Q[(i, i)] = -values[i] + lam * (weights[i]**2 - 2 * W * weights[i])

    # --- 物品 × 物品 交叉項 ---
    for i in range(n):
        for j in range(i + 1, n):
            Q[(i, j)] = 2 * lam * weights[i] * weights[j]

    # --- 鬆弛對角線 ---
    for k in range(M):
        idx = n + k
        Q[(idx, idx)] = lam * (4**k - 2 * W * 2**k)

    # --- 鬆弛 × 鬆弛 交叉項 ---
    for k in range(M):
        for l in range(k + 1, M):
            Q[(n + k, n + l)] = 2 * lam * 2**(k + l)

    # --- 物品 × 鬆弛 交叉項 ---
    for i in range(n):
        for k in range(M):
            Q[(i, n + k)] = 2 * lam * weights[i] * 2**k

    return Q, n, M

def decode_solution(sample, weights, values, n):
    """從 QUBO 解中提取物品選擇結果。"""
    selected = [i for i in range(n) if sample[i] == 1]
    tw = sum(weights[i] for i in selected)
    tv = sum(values[i] for i in selected)
    return selected, tw, tv



In [ ]:
# ============ Part (b): ExactSolver + lambda study ============
lambda_values = [1, 5, 10, 50]

print("\n=== Part (b): ExactSolver — lambda study ===")
print(f"{'lambda':>8} | {'Items':>20} | {'Weight':>6} | {'Value':>5} | {'Feasible':>8} | {'Optimal':>7}")
print("-" * 75)

for lam in lambda_values:
    Q, n_items, M = build_knapsack_qubo(weights, values, W, lam)
    bqm = dimod.BQM.from_qubo(Q)

    start = time.time()
    exact_result = dimod.ExactSolver().sample(bqm)
    t = time.time() - start

    sample = exact_result.first.sample
    selected, tw, tv = decode_solution(sample, weights, values, n_items)
    feasible = tw <= W
    optimal = (tv == best_value and feasible)

    items_str = str([i+1 for i in selected])
    print(f"{lam:>8} | {items_str:>20} | {tw:>6} | {tv:>5} | {str(feasible):>8} | {str(optimal):>7}")



=== Part (b): ExactSolver — lambda study ===
  lambda |                Items | Weight | Value | Feasible | Optimal
---------------------------------------------------------------------------
       1 |      [1, 2, 3, 4, 6] |    165 |   309 |     True |    True
       5 |      [1, 2, 3, 4, 6] |    165 |   309 |     True |    True
      10 |      [1, 2, 3, 4, 6] |    165 |   309 |     True |    True
      50 |      [1, 2, 3, 4, 6] |    165 |   309 |     True |    True


In [ ]:
import numpy as np
import time
from itertools import combinations
import dimod
import neal

# ============================================================
# 基本設定
# ============================================================
seed = 11412044
np.random.seed(seed)

weights = [23, 31, 29, 44, 53, 38, 63, 85, 89, 82]
values  = [92, 57, 49, 68, 60, 43, 67, 84, 87, 72]
W = 165
n = 10

# ============================================================
# QUBO 建構函數
# ============================================================
def build_knapsack_qubo(weights, values, W, lam):
    n = len(weights)
    M = int(np.ceil(np.log2(W)))  # = 8
    Q = {}

    # 物品對角線
    for i in range(n):
        Q[(i, i)] = -values[i] + lam * (weights[i]**2 - 2 * W * weights[i])

    # 物品×物品
    for i in range(n):
        for j in range(i + 1, n):
            Q[(i, j)] = 2 * lam * weights[i] * weights[j]

    # 鬆弛對角線
    for k in range(M):
        idx = n + k
        Q[(idx, idx)] = lam * (4**k - 2 * W * (2**k))

    # 鬆弛×鬆弛
    for k in range(M):
        for l in range(k + 1, M):
            Q[(n + k, n + l)] = 2 * lam * (2**(k + l))

    # 物品×鬆弛
    for i in range(n):
        for k in range(M):
            Q[(i, n + k)] = 2 * lam * weights[i] * (2**k)

    return Q, n, M

def decode_solution(sample, weights, values, n):
    """從 sample dict 提取物品選擇（已假設 value 是 Python int）"""
    selected = [i for i in range(n) if sample.get(i, 0) == 1]
    tw = sum(weights[i] for i in selected)
    tv = sum(values[i] for i in selected)
    return selected, tw, tv

# ====================================================================
#  把 sample 的 numpy int8 轉成 Python int
# ====================================================================
def to_pyint(sample):
    """Convert sample dict values from numpy.int8 → Python int."""
    return {k: int(v) for k, v in sample.items()}


In [ ]:
# ============================================================
# Part (b): ExactSolver — λ 效應研究
# ============================================================
print("\n" + "=" * 60)
print("Part (b): ExactSolver — Lambda Study")
print("=" * 60)

# ★ 使用跨越臨界點的 λ 值 ★
lambda_values = [0.01, 0.05, 0.1, 1, 10]

results_exact = {}

print(f"\n{'λ':>6} | {'Items (1-indexed)':>25} | {'Weight':>6} | {'Value':>5} "
      f"| {'Feasible':>8} | {'Optimal':>7} | {'Time(s)':>8}")
print("-" * 85)

for lam in lambda_values:
    Q, n_items, M = build_knapsack_qubo(weights, values, W, lam)
    bqm = dimod.BQM.from_qubo(Q)

    t_start = time.time()
    exact_result = dimod.ExactSolver().sample(bqm)
    t_exact = time.time() - t_start

    sample = to_pyint(exact_result.first.sample)
    energy = exact_result.first.energy

    selected, tw, tv = decode_solution(sample, weights, values, n_items)
    feasible = (tw <= W)
    optimal = (tv == best_value_classical and feasible)

    slack_val = sum((2**k) * sample.get(n_items + k, 0) for k in range(M))

    items_str = str([i + 1 for i in selected])
    print(f"{lam:>6} | {items_str:>25} | {tw:>6} | {tv:>5} "
          f"| {str(feasible):>8} | {str(optimal):>7} | {t_exact:>8.4f}")

    results_exact[lam] = {
        'items': selected, 'weight': tw, 'value': tv,
        'feasible': feasible, 'optimal': optimal,
        'energy': energy, 'time': t_exact, 'slack': slack_val
    }

# 分析結果
feasible_lams = [l for l in lambda_values if results_exact[l]['feasible']]
optimal_lams  = [l for l in lambda_values if results_exact[l]['optimal']]
infeasible_lams = [l for l in lambda_values if not results_exact[l]['feasible']]

print(f"\n✓ Lambda values yielding optimal solution: {optimal_lams}")
print(f"✗ Lambda values yielding infeasible solution: {infeasible_lams}")
print(f"  Theoretical critical point: λ_c ≈ 17/225 ≈ {17/225:.4f}")
print(f"  Heuristic rule (Appendix E): λ > max(v)/min(w) = {max(values)}/{min(weights)} = {max(values)/min(weights):.2f}")



Part (b): ExactSolver — Lambda Study

     λ |         Items (1-indexed) | Weight | Value | Feasible | Optimal |  Time(s)
-------------------------------------------------------------------------------------
  0.01 |        [1, 2, 3, 4, 5, 6] |    218 |   369 |    False |   False |   1.2892
  0.05 |           [1, 2, 3, 4, 5] |    180 |   326 |    False |   False |   0.9320
   0.1 |           [1, 2, 3, 4, 6] |    165 |   309 |     True |    True |   0.4211
     1 |           [1, 2, 3, 4, 6] |    165 |   309 |     True |    True |   0.4310
    10 |           [1, 2, 3, 4, 6] |    165 |   309 |     True |    True |   0.4929

✓ Lambda values yielding optimal solution: [0.1, 1, 10]
✗ Lambda values yielding infeasible solution: [0.01, 0.05]
  Theoretical critical point: λ_c ≈ 17/225 ≈ 0.0756
  Heuristic rule (Appendix E): λ > max(v)/min(w) = 92/23 = 4.00


In [ ]:
# ============================================================
# Part (c): Simulated Annealing — num_reads 研究
# ============================================================
print("\n" + "=" * 60)
print("Part (c): Simulated Annealing — num_reads Study")
print("=" * 60)

lam_sa = 10
Q_sa, n_items, M = build_knapsack_qubo(weights, values, W, lam_sa)
bqm_sa = dimod.BQM.from_qubo(Q_sa)

# ExactSolver 最佳 bitstring 作為參考
exact_ref = dimod.ExactSolver().sample(bqm_sa)
optimal_sample = to_pyint(exact_ref.first.sample)  # ★ 也要轉！
optimal_energy = exact_ref.first.energy

num_reads_list = [10, 100, 1000, 10000]
results_sa = {}

print(f"\n{'num_reads':>10} | {'Hits':>6} | {'Prob':>8} | {'Best Val':>8} "
      f"| {'Best Wt':>7} | {'Feasible':>8} | {'Time(s)':>8}")
print("-" * 75)

for nr in num_reads_list:
    t_start = time.time()
    sa_sampler = neal.SimulatedAnnealingSampler()
    sa_result = sa_sampler.sample(bqm_sa, num_reads=nr, seed=seed)
    t_sa = time.time() - t_start

    success_count = 0
    total_count = 0
    best_tv_sa = 0
    best_tw_sa = 0
    best_feasible = False

    for datum in sa_result.data():
        samp = to_pyint(datum.sample)  # ★ 每個 sample 都要轉！
        sel, tw, tv = decode_solution(samp, weights, values, n_items)
        num_occ = datum.num_occurrences
        total_count += num_occ

        if tw <= W and tv > best_tv_sa:
            best_tv_sa = tv
            best_tw_sa = tw
            best_feasible = True

        # 比對所有 18 個變數是否完全匹配
        match = all(samp.get(i, 0) == optimal_sample.get(i, 0)
                    for i in range(n_items + M))
        if match:
            success_count += num_occ

    prob = success_count / total_count if total_count > 0 else 0

    print(f"{nr:>10} | {success_count:>6} | {prob:>8.4f} | {best_tv_sa:>8} "
          f"| {best_tw_sa:>7} | {str(best_feasible):>8} | {t_sa:>8.4f}")

    results_sa[nr] = {
        'success': success_count, 'total': total_count, 'prob': prob,
        'best_value': best_tv_sa, 'best_weight': best_tw_sa,
        'feasible': best_feasible, 'time': t_sa
    }

# ============================================================
# Part (d): 比較表
# ============================================================
print("\n" + "=" * 60)
print("Part (d): Comparison Table")
print("=" * 60)

print(f"\n{'Method':<30} | {'Best Value':>10} | {'Weight':>6} | {'Feasible':>8} | {'Time(s)':>10}")
print("-" * 80)

print(f"{'Classical Brute Force':<30} | {best_value_classical:>10} | "
      f"{best_weight_classical:>6} | {'Yes':>8} | {time_classical:>10.6f}")

lam_report = 10
r = results_exact[lam_report]
feas_str = 'Yes' if r['feasible'] else 'No'
print(f"{'ExactSolver (λ='+str(lam_report)+')':<30} | {r['value']:>10} | "
      f"{r['weight']:>6} | {feas_str:>8} | {r['time']:>10.6f}")

for nr in num_reads_list:
    r = results_sa[nr]
    feas_str = 'Yes' if r['feasible'] else 'No'
    label = f"SA (reads={nr}, λ={lam_sa})"
    print(f"{label:<30} | {r['best_value']:>10} | "
          f"{r['best_weight']:>6} | {feas_str:>8} | {r['time']:>10.6f}")


Part (c): Simulated Annealing — num_reads Study

 num_reads |   Hits |     Prob | Best Val | Best Wt | Feasible |  Time(s)
---------------------------------------------------------------------------
        10 |      0 |   0.0000 |      284 |     161 |     True |   0.0122
       100 |      0 |   0.0000 |      284 |     161 |     True |   0.1320
      1000 |      5 |   0.0050 |      309 |     165 |     True |   0.7377
     10000 |     40 |   0.0040 |      309 |     165 |     True |   5.0259

Part (d): Comparison Table

Method                         | Best Value | Weight | Feasible |    Time(s)
--------------------------------------------------------------------------------
Classical Brute Force          |        309 |    165 |      Yes |   0.001797
ExactSolver (λ=10)             |        309 |    165 |      Yes |   0.492933
SA (reads=10, λ=10)            |        284 |    161 |      Yes |   0.012191
SA (reads=100, λ=10)           |        284 |    161 |      Yes |   0.131998
SA (reads